1.1 เตรียมสภาพแวดล้อมและข้อมูล

In [1]:
!pip install scikit-learn pandas numpy joblib --quiet

In [2]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import joblib
import os

1.2 เตรียมข้อมูล

In [3]:
# โหลดข้อมูลจาก URL โดยตรง
# dataset นี้เป็น public domain และใช้ได้อย่างเสรี
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
df = pd.read_csv(url)

# สำรวจข้อมูลเบื้องต้น — ขั้นตอนนี้สำคัญมาก
# อย่าข้ามไปแม้ว่าจะรีบ เพราะปัญหาในข้อมูลจะกลายเป็นปัญหาของโมเดล
print("รูปร่างของข้อมูล:", df.shape)
print("\n5 แถวแรก:")
print(df.head())
print("\nสถิติพื้นฐาน:")
print(df.describe())
print("\nจำนวน missing values:")
print(df.isnull().sum())

รูปร่างของข้อมูล: (768, 9)

5 แถวแรก:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

สถิติพื้นฐาน:
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   3

In [4]:
# แยก features และ target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# เก็บชื่อ features ไว้ เพราะเราจะต้องใช้ใน Streamlit app ด้วย
# การเก็บชื่อตรงนี้ช่วยให้แน่ใจว่า app ใช้ features ในลำดับเดียวกับที่ train
feature_names = X.columns.tolist()
print("Features ที่ใช้:", feature_names)

# แบ่งข้อมูลเป็น train และ test ด้วย random_state ที่กำหนดไว้
# random_state ทำให้ได้ผลเหมือนกันทุกครั้งที่รัน — สำคัญมากสำหรับ reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # ใช้ 20% สำหรับทดสอบ
    random_state=42,    # ตัวเลข 42 เป็นแค่ convention ไม่มีความพิเศษ
    stratify=y          # ทำให้สัดส่วน positive/negative ใน train/test เท่ากัน
)

print(f"\nจำนวนข้อมูล train: {len(X_train)} แถว")
print(f"จำนวนข้อมูล test: {len(X_test)} แถว")

Features ที่ใช้: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

จำนวนข้อมูล train: 614 แถว
จำนวนข้อมูล test: 154 แถว


1.3 สร้าง Pipeline

In [5]:
# สร้าง Pipeline ที่รวม preprocessing และ model เข้าด้วยกัน
# ลำดับของขั้นตอนใน Pipeline สำคัญมาก — ต้อง scale ก่อน train เสมอ
pipeline = Pipeline([
    # ขั้นที่ 1: StandardScaler ทำให้แต่ละ feature มี mean=0, std=1
    # สำคัญสำหรับ Random Forest น้อยกว่า SVM หรือ KNN
    # แต่ก็เป็น best practice ที่ควรทำเสมอ
    ("scaler", StandardScaler()),

    # ขั้นที่ 2: Random Forest Classifier พร้อม hyperparameters ที่เลือกแล้ว
    ("classifier", RandomForestClassifier(
        n_estimators=100,    # จำนวนต้นไม้ — มากขึ้นแม่นขึ้นแต่ช้าลง
        max_depth=6,         # ความลึกสูงสุด — ป้องกัน overfitting
        min_samples_split=5, # จำนวน samples ขั้นต่ำในการแบ่ง node
        random_state=42,     # reproducibility อีกครั้ง
        n_jobs=-1            # ใช้ CPU ทุก core ที่มี เพื่อความเร็ว
    ))
])

# train ทั้ง Pipeline — scikit-learn จะรัน scaler แล้วตามด้วย classifier ให้เอง
print("กำลัง train โมเดล...")
pipeline.fit(X_train, y_train)
print("Train เสร็จแล้ว!")

กำลัง train โมเดล...
Train เสร็จแล้ว!


1.4 ประเมินผลโมเดล

In [6]:
# ทำนายผลบน test set
y_pred = pipeline.predict(X_test)

# ดู accuracy โดยรวม
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy บน Test Set: {accuracy:.4f} ({accuracy*100:.2f}%)")

# classification_report ให้ข้อมูลที่ละเอียดกว่า accuracy เพียงอย่างเดียว
# Precision, Recall และ F1-Score ช่วยให้เข้าใจว่าโมเดลผิดพลาดแบบไหน
print("\nรายงานผลการทำนายแบบละเอียด:")
print(classification_report(y_test, y_pred,
                            target_names=["ไม่เป็นเบาหวาน", "เป็นเบาหวาน"]))

# ดู feature importance — Random Forest บอกได้ว่า feature ไหนสำคัญที่สุด
# ข้อมูลนี้มีคุณค่ามากทั้งสำหรับการปรับปรุงโมเดลและการอธิบายให้ผู้ใช้เข้าใจ
rf_model = pipeline.named_steps["classifier"]
importances = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\nความสำคัญของแต่ละ Feature:")
print(importances.to_string(index=False))

Accuracy บน Test Set: 0.7403 (74.03%)

รายงานผลการทำนายแบบละเอียด:
                precision    recall  f1-score   support

ไม่เป็นเบาหวาน       0.77      0.85      0.81       100
   เป็นเบาหวาน       0.66      0.54      0.59        54

      accuracy                           0.74       154
     macro avg       0.72      0.69      0.70       154
  weighted avg       0.73      0.74      0.73       154


ความสำคัญของแต่ละ Feature:
                 feature  importance
                 Glucose    0.360552
                     BMI    0.171625
                     Age    0.117001
DiabetesPedigreeFunction    0.097107
                 Insulin    0.072053
             Pregnancies    0.070756
           BloodPressure    0.056882
           SkinThickness    0.054025


1.5 บันทึกโมเดลและข้อมูลที่จำเป็น

In [7]:
# สร้างโฟลเดอร์สำหรับเก็บไฟล์โมเดล
os.makedirs("model_artifacts", exist_ok=True)

# บันทึก pipeline ทั้งหมดด้วย joblib
# joblib ดีกว่า pickle สำหรับ scikit-learn objects เพราะจัดการ numpy arrays ได้ดีกว่า
joblib.dump(pipeline, "model_artifacts/diabetes_pipeline.pkl")
print("บันทึก pipeline สำเร็จ!")

# บันทึก feature names แยกไว้ด้วย — สำคัญมาก!
# Streamlit app จะต้องรู้ว่าต้องรับ input อะไรบ้าง และลำดับต้องตรงกับที่ train
import json
with open("model_artifacts/feature_names.json", "w") as f:
    json.dump(feature_names, f)
print("บันทึก feature names สำเร็จ!")

# บันทึก model metadata — ข้อมูลนี้มีประโยชน์มากสำหรับ documentation
metadata = {
    "model_type": "RandomForestClassifier",
    "accuracy": round(float(accuracy), 4),
    "n_features": len(feature_names),
    "feature_names": feature_names,
    "target_names": ["ไม่เป็นเบาหวาน (0)", "เป็นเบาหวาน (1)"],
    "training_samples": len(X_train),
    "test_samples": len(X_test)
}
with open("model_artifacts/model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("บันทึก metadata สำเร็จ!")

# ตรวจสอบว่าไฟล์ถูกบันทึกครบถ้วน
print("\nไฟล์ที่บันทึกแล้ว:")
for f in os.listdir("model_artifacts"):
    size = os.path.getsize(f"model_artifacts/{f}")
    print(f"  {f} ({size:,} bytes)")

บันทึก pipeline สำเร็จ!
บันทึก feature names สำเร็จ!
บันทึก metadata สำเร็จ!

ไฟล์ที่บันทึกแล้ว:
  model_metadata.json (428 bytes)
  diabetes_pipeline.pkl (576,034 bytes)
  feature_names.json (113 bytes)


1.6 ทดสอบว่าโมเดลที่บันทึกไว้ใช้งานได้จริง

In [8]:
# โหลดโมเดลกลับมาใหม่ ราวกับว่าเราเป็น app ที่เพิ่ง start ขึ้นมา
loaded_pipeline = joblib.load("model_artifacts/diabetes_pipeline.pkl")

# สร้าง input ทดสอบที่เป็น case สมมติของผู้ป่วยรายหนึ่ง
# ลำดับของค่าต้องตรงกับ feature_names ที่บันทึกไว้
# [Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age]
test_input = np.array([[6, 148, 72, 35, 0, 33.6, 0.627, 50]])

# ทำนาย class (0 หรือ 1)
prediction = loaded_pipeline.predict(test_input)

# ทำนาย probability ของแต่ละ class
# predict_proba ส่งคืน array รูปแบบ [[prob_class_0, prob_class_1]]
probability = loaded_pipeline.predict_proba(test_input)

print(f"ผลการทำนาย: {'เป็นเบาหวาน' if prediction[0] == 1 else 'ไม่เป็นเบาหวาน'}")
print(f"ความน่าจะเป็น: {probability[0][1]*100:.1f}% ที่จะเป็นเบาหวาน")
print("\nโมเดลพร้อม deploy แล้ว ✓")

ผลการทำนาย: เป็นเบาหวาน
ความน่าจะเป็น: 64.8% ที่จะเป็นเบาหวาน

โมเดลพร้อม deploy แล้ว ✓


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [9]:
import logging
import time
from datetime import datetime

# ตั้งค่าระบบ logging — ทำครั้งเดียวตอนเริ่ม application
logging.basicConfig(
    level=logging.INFO,
    # format นี้ทำให้ log อ่านง่ายและมีข้อมูลครบ
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler("ml_service.log"),  # บันทึกลงไฟล์
        logging.StreamHandler()                  # แสดงบน console ด้วย
    ]
)
logger = logging.getLogger(__name__)

def predict_with_logging(features, user_id):
    start_time = time.time()
    logger.info(f"รับ prediction request จาก user_id={user_id}")

    try:
        # ตรวจสอบความสมเหตุสมผลของ input ก่อนส่งเข้าโมเดล
        if features["age"] < 0 or features["age"] > 120:
            # Warning เพราะระบบยังทำงานต่อได้ แต่ข้อมูลดูแปลก
            logger.warning(f"ค่า age={features['age']} ผิดปกติ user_id={user_id}")

        prediction = model.predict([list(features.values())])[0]
        elapsed = time.time() - start_time

        # บันทึก prediction และเวลาที่ใช้ — ข้อมูลนี้มีคุณค่าสำหรับ monitoring
        logger.info(f"ทำนายสำเร็จ | result={prediction} | เวลา={elapsed:.3f}s | user={user_id}")
        return prediction

    except Exception as e:
        # บันทึก error พร้อม context ครบถ้วน เพื่อให้ debug ได้ในภายหลัง
        logger.error(f"เกิดข้อผิดพลาด | error={str(e)} | user={user_id} | features={features}")
        raise  # โยน error ต่อให้ caller จัดการ ไม่ "กลืน" error ไว้เงียบๆ